In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Organizing and interpolating population and poverty data
https://www.nhgis.org/

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import numpy as np
from pathlib import Path


%matplotlib inline
# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker
import seaborn as sns

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {#'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Times'],
              #'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction": "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

In [ ]:
from scipy import interpolate
def get_x_y_inter_extrapolation(x,y,min_year, max_year,kind = 'quadratic'):
    x = np.array(x.tolist())
    y = np.array(y.tolist())

    new_y=np.zeros(max_year-min_year)
    new_x=np.arange(min_year,max_year)
    f = interpolate.interp1d(x, np.log(y),fill_value = "extrapolate",kind=kind)
    new_y=np.exp(f(new_x))
   
    return new_x, new_y

def interpolate_dataframe(df,min_year,max_year,kind='quadratic'):
    values=[]
    x=df.columns[[str(e).isnumeric() for e in df.columns]]    
    for index in df.index:
        y=df[x].iloc[index].values
        newx,newy=get_x_y_inter_extrapolation(x,y,min_year,max_year,kind)
        values.append(newy)
    values=np.transpose(values)
    columns=['GISJOIN','STATE','STATEFP','STATENH','COUNTY', 'COUNTYFP','COUNTYNH']

    interpolation_df=df[columns].copy()
    for year in range(min_year,max_year):
        interpolation_df[year]=values[year-min_year]
    return interpolation_df

def fix_miami_dade(df,columns_not_includes=['GISJOIN', 'STATE', 'STATEFP', 'STATENH', 'COUNTY', 'COUNTYFP','COUNTYNH']):
    columns=list(df.columns)
    for c in columns_not_includes:
        columns.remove(c)


    df1=df[df['GISJOIN']=='G1200250'][columns].fillna(0).values.astype(float)
    df2=df[df['GISJOIN']=='G1200860'][columns].fillna(0).values.astype(float)
    values=(df1+df2)[0]

    index=df[df['GISJOIN']=='G1200860'].index[0]
    for i,c in enumerate(columns):
        df.at[index,c]=values[i]

    df=df[~df['COUNTY'].isna()].reset_index(drop=True)
    return df

def remove_dade_county(df):
    df = df.drop(df[df['GISJOIN']=='G1200250'].index[0]).reset_index(drop=True)
    return df

In [ ]:
glob.glob('../data/ipumns_raw_data/*/' )

# Population by race

## Census data

In [ ]:
def read_excel_file(file):
    df = pd.read_excel(file,skiprows=5)
    df = df.drop(0)
    return df

def parse_excel_file(df):
    df['Population'] = df[df.columns[3:]].sum(axis=1)
    df['FIPS'] = df['FIPS State and County Codes'].apply(lambda x: str(int(x)).zfill(5))
    df['Year'] = df['Year of Estimate'].astype(int)
    df['Race'] = df['Race/Sex Indicator'].str.replace(' female','')
    df['Race'] = df['Race'].str.replace(' male','')
    df = df[['Year', 'FIPS', 'Race','Population']]
    df = df.groupby(['Year','FIPS','Race']).sum().reset_index()
    return df

def read_all_excel_files(files):
    dataframes = []
    for file in files:
        df = read_excel_file(file)
        df = parse_excel_file(df)
        dataframes.append(df)

    df_group = pd.concat(dataframes)
    return df_group

def transpose_excel_data(df_group):
    blacks_df     = df_group[df_group.Race=='Black'][['Year','FIPS','Population']].rename(columns={'Population':'Blacks'})
    whites_df     = df_group[df_group.Race=='White'][['Year','FIPS','Population']].rename(columns={'Population':'Whites'})
    minorities_df = df_group[df_group.Race=='Other races'][['Year','FIPS','Population']].rename(columns={'Population':'Minorities'})
    dfgm = pd.merge(blacks_df,whites_df,on=['FIPS','Year'],how='outer') 
    dfgm = pd.merge(dfgm,minorities_df,on=['FIPS','Year'],how='outer')
    dfgm['Total'] = dfgm['Blacks']+dfgm['Whites']+dfgm['Minorities']
    return dfgm


In [ ]:
files = glob.glob('../data/county_population_1970_1989_us_census/*.xls')
df_group1 = read_all_excel_files(files)
dfgm1 = transpose_excel_data(df_group1)

In [ ]:
dfgm1.sort_values(by='Year')

## CDC data

In [ ]:
def read_and_parse_data_cdc(file):
    df = pd.read_csv(file,sep='\t')
    df = df[~df['County'].isna()]
    for i in df[df['Notes']=='Total'].index:
        df.at[i,'Race'] = 'Total'
    df['FIPS'] = df['County Code'].apply(lambda x: str(int(x)).zfill(5))
    df = df[['County','FIPS','Race','Population']]
    map_names = {'American Indian or Alaska Native':'Minorities', 'Asian or Pacific Islander':'Minorities','Black or African American':'Blacks', 'Total':'Total', 'White':'Whites'}
    df['Race'] = df['Race'].map(map_names)
    df['Year'] = int(file.split('/')[-1].strip('.txt'))
    df = df.groupby(['Year','County','FIPS','Race']).sum().reset_index()
    return df

def parse_and_concat_all_cdc_files(files):
    data = []
    for file in files:
        data.append(read_and_parse_data_cdc(file))
    df_group = pd.concat(data)
    df_group = df_group.sort_values(by='Year').reset_index(drop=True)
    return df_group[['Year','FIPS','Race','Population']]

def transpose_cdc_data(df_group):
    blacks_df     = df_group[df_group.Race=='Blacks'][['Year','FIPS','Population']].rename(columns={'Population':'Blacks'})
    whites_df     = df_group[df_group.Race=='Whites'][['Year','FIPS','Population']].rename(columns={'Population':'Whites'})
    minorities_df = df_group[df_group.Race=='Minorities'][['Year','FIPS','Population']].rename(columns={'Population':'Minorities'})
    minorities_df
    total_df =    df_group[df_group.Race=='Total'][['Year','FIPS','Population']].rename(columns={'Population':'Total'})

    dfgm = pd.merge(total_df,blacks_df,on=['FIPS','Year'],how='outer') 
    dfgm = pd.merge(dfgm,whites_df,on=['FIPS','Year'],how='outer')
    dfgm = pd.merge(dfgm,minorities_df,on=['FIPS','Year'],how='outer')

    return dfgm

def organize_df(dfgm,var='Blacks'):
    df = dfgm[['Year','FIPS',var]]
    dfm = df[df.Year==1970][['FIPS',var]].rename(columns={var:1970})
    for year in range(1971,2020):
        dfl = df[df.Year==year][['FIPS',var]].rename(columns={var:year})
        dfm = pd.merge(dfm,dfl,on=['FIPS'],how='outer')
    dfm = dfm.fillna(0)
    return dfm

In [ ]:
files = glob.glob('../data/cdc_county_population_1990_2019/*.txt')
df_group2 = parse_and_concat_all_cdc_files(files)
dfgm2 = transpose_cdc_data(df_group2)

In [ ]:
min(dfgm1["Year"]),max(dfgm1['Year'])

In [ ]:
min(dfgm2["Year"]),max(dfgm2['Year'])

In [ ]:
dfgm = pd.concat([dfgm1,dfgm2])

In [ ]:
blacks_df= organize_df(dfgm,var='Blacks')
blacks_df['FIPS'] = blacks_df['FIPS'].apply(lambda x: str(int(x)).zfill(5))
whites_df= organize_df(dfgm,var='Whites')
whites_df['FIPS'] = whites_df['FIPS'].apply(lambda x: str(int(x)).zfill(5))
minorities_df= organize_df(dfgm,var='Minorities')
minorities_df['FIPS'] = minorities_df['FIPS'].apply(lambda x: str(int(x)).zfill(5))
population_df= organize_df(dfgm,var='Total')
population_df['FIPS'] = population_df['FIPS'].apply(lambda x: str(int(x)).zfill(5))

In [ ]:
pop_df = pd.read_csv('../data/ipumns_raw_data/population_1970-2010/nhgis0003_ts_nominal_county.csv')
pop_df.head()

In [ ]:
df = pop_df[['GISJOIN',    'STATE',  'STATEFP',  'STATENH',   'COUNTY', 'COUNTYFP',
       'COUNTYNH']]
df['COUNTYFP'] = df['COUNTYFP'].apply(lambda x: str(int(x)).zfill(3))
df['STATEFP'] = df['STATEFP'].apply(lambda x: str(int(x)).zfill(2))
df['FIPS'] = df['STATEFP'] + df['COUNTYFP']

merged_population_df = pd.merge(population_df,df, on = 'FIPS',how='outer')
merged_blacks_df = pd.merge(blacks_df,df, on = 'FIPS',how='outer')
merged_whites_df = pd.merge(whites_df,df, on = 'FIPS',how='outer')
merged_minorities_df = pd.merge(minorities_df,df, on = 'FIPS',how='outer')

In [ ]:
nonwhite_df = pd.DataFrame()
for column in merged_minorities_df.columns:
    if str(column).isnumeric():
        nonwhite_df[column]= merged_blacks_df[column]+merged_minorities_df[column]
    else:
        nonwhite_df[column]= merged_minorities_df[column]

In [ ]:
df1960

In [ ]:
path1960=glob.glob('../data/ipumns_raw_data/race_1960/*.csv')[0]
df1960=pd.read_csv(path1960)
df1960=df1960[['GISJOIN','B48001','B48002']]
df1960.rename(columns={'B48001':1960,'B48002':'Non-White'},inplace=True)
merged_whites_df = pd.merge(merged_whites_df,df1960[['GISJOIN',1960]],on='GISJOIN',how = 'outer')
merged_whites_df.head()

In [ ]:
merged_nonwhite_df = pd.merge(nonwhite_df,df1960[['GISJOIN','Non-White']],on='GISJOIN',how = 'outer')
merged_nonwhite_df.rename(columns = {'Non-White':1960},inplace=True)
merged_nonwhite_df.head()

In [ ]:
path1960=glob.glob('../data/ipumns_raw_data/race_1960/*.csv')[0]
df1960=pd.read_csv(path1960)
df1960=df1960[['GISJOIN','B48001','B48002']]
df1960.rename(columns={'B48001':'White','B48002':'Non-White'},inplace=True)
df1960['Total'] = df1960['White'] + df1960['Non-White']
df1960

In [ ]:
path1960=glob.glob('../data/ipumns_raw_data/population_1960/*.csv')[0]
df1960=pd.read_csv(path1960)[['GISJOIN','B47001']]
df1960.rename(columns={'B47001':1960},inplace=True)
merged_population_df = pd.merge(merged_population_df,df1960[['GISJOIN',1960]],on='GISJOIN',how = 'outer')
merged_population_df.head()

In [ ]:
merged_population_df = merged_population_df[~merged_population_df['COUNTY'].isna()]

In [ ]:
merged_population_df[merged_population_df['COUNTY'].str.contains('Miami')]

In [ ]:
merged_population_df[merged_population_df['COUNTY'].str.contains('Dade')]

In [ ]:
merged_population_df[merged_population_df['GISJOIN']=='G1200250']

In [ ]:
merged_population_df[merged_population_df['GISJOIN']=='G1200860']

### Fix Miami-Dade name 

In [ ]:
merged_population_df = fix_miami_dade(merged_population_df)
merged_blacks_df = fix_miami_dade(merged_blacks_df)
merged_whites_df = fix_miami_dade(merged_whites_df)
merged_minorities_df = fix_miami_dade(merged_minorities_df)

merged_population_df = remove_dade_county(merged_population_df)
merged_blacks_df = remove_dade_county(merged_blacks_df)
merged_whites_df = remove_dade_county(merged_whites_df)
merged_minorities_df = remove_dade_county(merged_minorities_df)


In [ ]:
merged_population_df


## Time series visualization 

In [ ]:
xs = 10
ys = 10

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
df= merged_population_df[[i for i in range(1970,2017)]]

for city in range(0,len(merged_population_df)):
    x=[i for i in range(1970,2017)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= merged_blacks_df[[i for i in range(1970,2017)]]

for city in range(0,len(merged_blacks_df)):
    x=[i for i in range(1970,2017)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Black population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
import ipywidgets as widMgets
from ipywidgets import interact

In [ ]:
merged_blacks_df[merged_blacks_df.COUNTY.str.contains('Orleans')]

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual


In [ ]:
names = (merged_blacks_df['COUNTY']+', '+merged_blacks_df['STATE']).values
@interact(i=(0,len(merged_blacks_df)-1,1))
def plot_pop_interact(i=1144):
    df= merged_blacks_df[[i for i in range(1970,2017)]]

    fig, ax = plt.subplots(figsize=(xs,ys))

    x=[i for i in range(1970,2017)]
    y=df.iloc[i]
    ax.plot(x,y,'o-',color='k',linewidth=2,alpha=0.6)
    ax.set_ylabel(r"Black population",multialignment='center')
    ax.set_xlabel(r"Year, $t$")
    ax.set_title(names[i]+' - {}'.format(i))
    fig.show()

    

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= merged_whites_df[[i for i in range(1970,2017)]]

for city in range(0,len(merged_whites_df)):
    x=[i for i in range(1970,2017)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"White population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= merged_minorities_df[[i for i in range(1970,2017)]]

for city in range(0,len(merged_minorities_df)):
    x=[i for i in range(1970,2017)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Minorities population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

# Poverty

In [ ]:
df_census_poverty = pd.read_excel('../data/Poverty-Rates-by-County-1960-2010.xlsm',sheet_name = 'Data',skiprows=1)
df_census_poverty = df_census_poverty.iloc[0:3194]
df_census_poverty = df_census_poverty[~df_census_poverty['Number'].isna()]
df_census_poverty['FIPS'] = df_census_poverty['FIPS'].apply(lambda x: str(int(x)).zfill(5))
df_census_poverty.head()

In [ ]:
df_census_poverty.at[372,'Persons in Poverty 1960']=int(22.29/100*935047.0)

In [ ]:
df_census_poverty = df_census_poverty[['FIPS','Persons in Poverty 1960']]

In [ ]:
df_census_poverty.head()

## Census poverty data 

In [ ]:
fields = [[1,2,'FIPS State code'],
          [4,6,'FIPS county code'],
          [8,15,'Poverty'],#'Estimate of people of all ages in poverty'],
          [17,24,'90% confidence interval lower bound of estimate of people of all ages in poverty'],
          [26,33,'90% confidence interval upper bound of estimate of people of all ages in poverty'],
          [35,38,'Estimated percent of people of all ages in poverty'],
          [40,43,'90% confidence interval lower bound of estimate of percent of people of all ages in poverty'],
          [45,48,'90% confidence interval upper bound of estimate of percent of people of all ages in poverty'],
          [50,57,'Estimate of people age 0-17 in poverty'],
          [59,66,'90% confidence interval lower bound of estimate of people age 0-17 in poverty'],
          [68,75,'90% confidence interval upper bound of estimate of people age 0-17 in poverty'],
          [77,80,'Estimated percent of people age 0-17 in poverty'],
          [82,85,'90% confidence interval lower bound of estimate of percent of people age 0-17 in poverty'],
          [87,90,'90% confidence interval upper bound of estimate of percent of people age 0-17 in poverty'],
          [92,99,'Estimate of related children age 5-17 in families in poverty'],
          [101,108,'90% confidence interval lower bound of estimate of related children age 5-17 in families in poverty'],
          [110,117,'90% confidence interval upper bound of estimate of related children age 5-17 in families in poverty'],
          [119,122,'Estimated percent of related children age 5-17 in families in poverty'],
          [124,127,'90% confidence interval lower bound of estimate of percent of related children age 5-17 in families in poverty'],
          [129,132,'90% confidence interval upper bound of estimate of percent of related children age 5-17 in families in poverty'],
          [134,139,'Estimate of median household income'],
          [141,146,'90% confidence interval lower bound of estimate of median household income'],
          [148,153,'90% confidence interval upper bound of estimate of median household income'],
          [155,161,'Estimate of people under age 5 in poverty'],
          [163,169,'90% confidence interval lower bound of estimate of people under age 5 in poverty'],
          [171,177,'90% confidence interval upper bound of estimate of people under age 5 in poverty'],
          [179,182,'Estimated percent of people under age 5 in poverty'],
          [184,187,'90% confidence interval lower bound of estimate of percent of people under age 5 in poverty'],
          [189,192,'90% confidence interval upper bound of estimate of percent of people under age 5 in poverty'],
          [194,238,'Name'],
          [240,241,'State']
         ]

In [ ]:
files = glob.glob('../data/census_poverty_data_1989_2019/*')

In [ ]:
def get_line_data(line):
    line_data=[]
    for field in fields: 
        line_data.append(line[field[0]-1:field[1]].strip(' \n'))
    return line_data

def parse_poverty_data(file):
    data = file.readlines()

    parsed_data=[]
    for line in data:
        line_data = get_line_data(line)
        parsed_data.append(line_data)

    df = pd.DataFrame(parsed_data,columns=np.transpose(fields)[-1])
    return df

In [ ]:
for year in range(1989,2003):
    if year not in [1990,1991,1992,1994,1996]:
        file = open('../data/census_poverty_data_1989_2019/{}.dat'.format(year))
        df = parse_poverty_data(file)
        df.to_csv('../data/census_poverty_data_1989_2019/{}.csv'.format(year))

In [ ]:
data = []
for year in range(1989,2003):
    if year not in [1990,1991,1992,1994,1996,2000]:
        df = pd.read_csv('../data/census_poverty_data_1989_2019/{}.csv'.format(year),index_col=0)
        df['FIPS State code'] = df['FIPS State code'].apply(lambda x: str(x).zfill(2))
        df['FIPS county code'] = df['FIPS county code'].apply(lambda x: str(x).zfill(3))
        df['FIPS'] = df['FIPS State code'] + df['FIPS county code']
        df = df[['FIPS','Poverty']]
        df.rename(columns={'Poverty':year},inplace=True)
        data.append(df)
        
for year in range(2003,2020):
#     print(year)
    if year<2005:
        df=pd.read_excel('../data/census_poverty_data_1989_2019/{}.xls'.format(year),skiprows=1)
  
    elif year<2013:
        df=pd.read_excel('../data/census_poverty_data_1989_2019/{}.xls'.format(year),skiprows=2)
    else:
        df=pd.read_excel('../data/census_poverty_data_1989_2019/{}.xls'.format(year),skiprows=3)

    if year < 2012:
        df['State FIPS'] = df['State FIPS'].apply(lambda x: str(x).replace('.0','').zfill(2))
        df['County FIPS'] = df['County FIPS'].apply(lambda x: str(x).replace('.0','').zfill(3))
        df['FIPS'] = df['State FIPS'] + df['County FIPS']
        df = df[['FIPS','Poverty Estimate All Ages']]
        df.rename(columns={'Poverty Estimate All Ages':year},inplace=True)
    else:
        df['State FIPS Code'] = df['State FIPS Code'].apply(lambda x: str(x).replace('.0','').zfill(2))
        df['County FIPS Code'] = df['County FIPS Code'].apply(lambda x: str(x).replace('.0','').zfill(3))
        df['FIPS'] = df['State FIPS Code'] + df['County FIPS Code']
        df = df[['FIPS','Poverty Estimate, All Ages']]
        df.rename(columns={'Poverty Estimate, All Ages':year},inplace=True)

    data.append(df)

In [ ]:
df=data[0]
for i in range(1,len(data)):
    data[i][data[i].columns[1]]=data[i][data[i].columns[1]].replace('.',np.nan)
    data[i][data[i].columns[1]]=data[i][data[i].columns[1]].astype(float)
    df = pd.merge(df,data[i],on=['FIPS'],how='outer')


## IPUMS data

In [ ]:
glob.glob('../data/ipumns_raw_data/*/' )

In [ ]:
poverty_df=pd.read_csv(glob.glob('../data/ipumns_raw_data/poverty/*.csv')[0])
poverty_df.rename(columns={'CL6AA1970':1970,'CL6AA1980':1980,'CL6AA1990':1990,
                           'CL6AA2000':2000,'CL6AA125':2010},inplace=True)
del poverty_df['CL6AA125M']
del poverty_df[2010]
poverty_df['STATEFP'] = poverty_df['STATEFP'].apply(lambda x: str(x).zfill(2))
poverty_df['COUNTYFP'] = poverty_df['COUNTYFP'].apply(lambda x: str(x).zfill(3))
poverty_df['FIPS'] = poverty_df['STATEFP'] + poverty_df['COUNTYFP']
poverty_df.head()

In [ ]:
df_poverty = pd.merge(poverty_df,df,on='FIPS',how='inner')
df_poverty = df_poverty.replace('.',np.nan)
df_poverty.head()

In [ ]:
selected_columns=poverty_df.columns

In [ ]:
df_poverty = pd.merge(df_poverty,df_census_poverty[['FIPS','Persons in Poverty 1960']],on='FIPS',how = 'outer')
df_poverty.rename(columns = {'Persons in Poverty 1960':1960},inplace=True)

In [ ]:
df_poverty = df_poverty[~df_poverty['GISJOIN'].isna()]

In [ ]:
df_poverty=fix_miami_dade(df_poverty)

df_poverty[df_poverty.COUNTY.str.contains('Miami')]

In [ ]:
for year in [ 1960, 1970,       1980,       1990,       2000,     
             1989,       1993,       1995,       1997,       1998,       1999,
             2001,       2002,       2003,       2004,       2005,       2006,
             2007,       2008,       2009,       2010,       2011,       2012,
             2013,       2014,       2015,       2016,       2017,       2018,
             2019]:
    df_poverty[year] = df_poverty[year].astype(float)

In [ ]:
df_poverty[df_poverty.COUNTY.str.contains('Miami')][ [ 1960, 1970,       1980,       1990,       2000,     
             1989,       1993,       1995,       1997,       1998,       1999,
             2001,       2002,       2003,       2004,       2005,       2006,
             2007,       2008,       2009,       2010,       2011,       2012,
             2013,       2014,       2015,       2016,       2017,       2018,
             2019]].loc[367]

# Education 

In [ ]:
glob.glob('../data/ipumns_raw_data/education/*' )

In [ ]:
selected_columns=list(selected_columns)

In [ ]:
education_df=pd.read_csv('../data/ipumns_raw_data/education/nhgis0014_ts_nominal_county.csv')
education_df.rename(columns={'B69AA1970':1970,
                             'B69AA1980':1980,
                             'B69AA1990':1990,
                             'B69AA2000':2000,
                             'B69AA125':2010},inplace=True)
education_df=education_df[['GISJOIN','STATE','STATEFP','STATENH','COUNTY','COUNTYFP','COUNTYNH',1970,1980,1990,2000,2010]
]
education_df.head()

# Unemployment 

In [ ]:
education_df=fix_miami_dade(education_df)

education_df[education_df.COUNTY.str.contains('Miami')]

## Labor department 

In [ ]:
def parse_unemployment_data(file):
    file_data = file.readlines()[6:-3]
    limits =[0,16,21,29,80,86,100,115,125,132]
    unemployment_data=[]
    for line in file_data:
        line_data = []
        for i in range(len(limits)-1):
            line_data.append(line[limits[i]:limits[i+1]+1].strip())
        unemployment_data.append(line_data)
    df = pd.DataFrame(unemployment_data,columns=['LAUS','State FIPS','County FIPS','County','Year','Labor Force','Employed','Unemployed','Unemployed rate'])
    return df

In [ ]:
def read_unemployment_dataframe(year):
    df = pd.read_csv('../data/annual_unemployment_1990-2020/laucnty{}.csv'.format(year),index_col=0)
    df['State FIPS'] = df['State FIPS'].apply(lambda x: str(x).zfill(2))
    df['County FIPS'] = df['County FIPS'].apply(lambda x: str(x).zfill(3))
    df['FIPS'] = df['State FIPS'] + df['County FIPS']
    return df

def select_and_cast_unemployment_data(df,year):
    df = df[['FIPS','Unemployed']]
    df['Unemployed']=df['Unemployed'].replace('N.A.',np.nan)
    df['Unemployed']=df['Unemployed'].str.replace(',','')
    df['Unemployed']=df['Unemployed'].astype(float)
    df.rename(columns = {'Unemployed':year},inplace=True)
    return df

In [ ]:
for year in range(1990,2020):
    file = open('../data/annual_unemployment_1990-2020/laucnty{}.txt'.format(year))
    df = parse_unemployment_data(file)
    df.to_csv('../data/annual_unemployment_1990-2020/laucnty{}.csv'.format(year))

In [ ]:
year=1990
df = read_unemployment_dataframe(year)
unemployment_data = select_and_cast_unemployment_data(df,year)
for year in range(1991,2020):
    df = read_unemployment_dataframe(year)
    df = select_and_cast_unemployment_data(df,year)
    unemployment_data = pd.merge(unemployment_data,df,on="FIPS",how='outer')

In [ ]:
unemployment_data.head()

## IPUMS 

In [ ]:
glob.glob('../data/ipumns_raw_data/labor_force_and_employment/*' )

In [ ]:
selected_columns

In [ ]:
unemployment_df=pd.read_csv('../data/ipumns_raw_data/labor_force_and_employment/nhgis0015_ts_nominal_county.csv' )
unemployment_df.rename(columns={'B84AE1970':1970,
                              'B84AE1980':1980,
                              'B84AE1990':1990,
                              'B84AE2000':2000,
                              'B84AE125':2010},inplace=True)

unemployment_df=unemployment_df[['GISJOIN','STATE','STATEFP','STATENH','COUNTY','COUNTYFP','COUNTYNH',1970,1980]]#1990,2000,2010]]
unemployment_df.head()

In [ ]:
unemployment_df=fix_miami_dade(unemployment_df)
unemployment_df.reset_index(inplace=True,drop=True)
unemployment_df['STATEFP'] = unemployment_df['STATEFP'].apply(lambda x: str(x).zfill(2))
unemployment_df['COUNTYFP'] = unemployment_df['COUNTYFP'].apply(lambda x: str(x).zfill(3))
unemployment_df['FIPS'] = unemployment_df['STATEFP'] + unemployment_df['COUNTYFP']
unemployment_df[unemployment_df.COUNTY.str.contains('Miami')]

In [ ]:
unemployment_df_merged = pd.merge(unemployment_df,unemployment_data,on='FIPS',how='outer')
unemployment_df_merged.head()

# Interpolation

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression


In [ ]:
from scipy import interpolate

def get_x_y_inter_extrapolation(x,y,min_year, max_year,kind = 'linear'):
    x = np.array(x.tolist())
    y = np.array(y.tolist())

    new_y=np.zeros(max_year-min_year)
    new_x=np.arange(min_year,max_year)
    f = interpolate.interp1d(x, np.log1p(y),fill_value = "extrapolate",kind=kind)
#     f = interpolate.interp1d(x, y,fill_value = "extrapolate",kind=kind)
    new_y=np.expm1(f(new_x))
    return new_x, new_y

def interpolate_dataframe(df,min_year,max_year,kind='quadratic'):
    values=[]
    x=df.columns[[str(e).isnumeric() for e in df.columns]]    
    for index in df.index:
        y=df[x].iloc[index].values
        newx,newy=get_x_y_inter_extrapolation(x,y,min_year,max_year,kind)
        values.append(newy)
    values=np.transpose(values)
    columns=['GISJOIN','STATE','STATEFP','STATENH','COUNTY', 'COUNTYFP','COUNTYNH']

    interpolation_df=df[columns].copy()
    for year in range(min_year,max_year):
        interpolation_df[year]=values[year-min_year]
    return interpolation_df

## 1970 - 2021 

In [ ]:
min_year = 1970
max_year = 2021
kind = 'linear'
pop_interpolation_df=interpolate_dataframe(merged_population_df,min_year,max_year,kind=kind)
pop_interpolation_df.to_csv('../data/clean_dataframes/total_population.csv')

whites_df_interpolation=interpolate_dataframe(merged_whites_df,min_year,max_year,kind=kind)
whites_df_interpolation.to_csv('../data/clean_dataframes/white_population.csv')

blacks_df_interpolation=interpolate_dataframe(merged_blacks_df,min_year,max_year,kind=kind)
blacks_df_interpolation.to_csv('../data/clean_dataframes/black_population.csv')
# 
minorities_df_interpolation=interpolate_dataframe(merged_minorities_df,min_year,max_year,kind=kind)
minorities_df_interpolation.to_csv('../data/clean_dataframes/minorities_population.csv')

poverty_df_interpolation=interpolate_dataframe(df_poverty,min_year,max_year,kind=kind)
poverty_df_interpolation.to_csv('../data/clean_dataframes/poverty_population.csv')

education_df_interpolation=interpolate_dataframe(education_df,min_year,max_year,kind=kind)
education_df_interpolation.to_csv('../data/clean_dataframes/education.csv')

unemployment_df=interpolate_dataframe(unemployment_df_merged,min_year,max_year,kind = kind)
unemployment_df.to_csv('../data/clean_dataframes/unemployment.csv')

In [ ]:

def plot_pop_interact(dfpop,k,year_min=1970,year_max=2021):
    df = dfpop[[i for i in range(year_min,year_max)]]

    fig, ax = plt.subplots(figsize=(xs,ys))

    x=[i for i in range(year_min,year_max)]
    y=df.iloc[i]
    ax.plot(x,y,'o-',color='k',linewidth=2,alpha=0.6)
    ax.set_ylabel(r"Y(t)",multialignment='center')
    ax.set_xlabel(r"$t$")
    ax.set_title(names[k]+' - {}'.format(k))
    fig.show()

    

In [ ]:
plot_pop_interact(pop_interpolation_df,k=606)

In [ ]:
plot_pop_interact(whites_df_interpolation,k=606)

In [ ]:
plot_pop_interact(blacks_df_interpolation,k=606)

In [ ]:
plot_pop_interact(minorities_df_interpolation,k=606)

In [ ]:
plot_pop_interact(poverty_df_interpolation,k=606)

In [ ]:
plot_pop_interact(education_df_interpolation,k=606)

In [ ]:
plot_pop_interact(unemployment_df,k=606)

## 1960-2021

In [ ]:
min_year = 1960
max_year = 2021
kind = 'linear'

pop_interpolation_df=interpolate_dataframe(merged_population_df,min_year,max_year,kind=kind)
pop_interpolation_df.to_csv('../data/clean_dataframes/1960-2021/total_population.csv')

whites_df_interpolation=interpolate_dataframe(merged_whites_df,min_year,max_year,kind=kind)
whites_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/white_population.csv')

blacks_df_interpolation=interpolate_dataframe(merged_blacks_df,min_year,max_year,kind=kind)
blacks_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/black_population.csv')

minorities_df_interpolation=interpolate_dataframe(merged_minorities_df,min_year,max_year,kind=kind)
minorities_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/minorities_population.csv')

nonwhite_df_interpolation=interpolate_dataframe(merged_nonwhite_df,min_year,max_year,kind=kind)
nonwhite_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/nonwhite_population.csv')

poverty_df_interpolation=interpolate_dataframe(df_poverty,min_year,max_year,kind=kind)
poverty_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/poverty_population.csv')

education_df_interpolation=interpolate_dataframe(education_df,min_year,max_year,kind=kind)
education_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/education.csv')

unemployment_df_interpolation=interpolate_dataframe(unemployment_df_merged,min_year,max_year,kind = kind)
unemployment_df_interpolation.to_csv('../data/clean_dataframes/1960-2021/unemployment.csv')

In [ ]:
plot_pop_interact(nonwhite_df_interpolation,k=10,year_min=1960,year_max=2021)

In [ ]:
plot_pop_interact(nonwhite_df_interpolation,k=1,year_min=1960,year_max=2021)

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= nonwhite_df_interpolation[[i for i in range(1960,2020)]]

for city in range(0,len(merged_population_df)):
    x=[i for i in range(1960,2020)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Non-white Population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= unemployment_df_interpolation[[i for i in range(1960,2020)]]

for city in range(0,len(merged_population_df)):
    x=[i for i in range(1960,2020)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Unemployment",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= poverty_df_interpolation[[i for i in range(1960,2020)]]

for city in range(0,len(merged_population_df)):
    try:
        x=[i for i in range(1960,2020)]
        y=df.iloc[city]
        ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
    except:
        continue
ax.set_ylabel(r"Poverty",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= education_df_interpolation[[i for i in range(1960,2020)]]

for city in range(0,len(merged_population_df)):
    x=[i for i in range(1960,2020)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Education",multialignment='center')
ax.set_xlabel(r"Year, $t$")